# 04 — Algorithmic Customer Segmentation & Business Translation

**Author:** Person B (RFM & Customer Segmentation Lead)  
**Project:** Online Retail Customer Analytics & Segmentation  
**Upstream Input:** `data/rfm_preprocessed.parquet` & `data/rfm_features.parquet` (Outputs from Notebook 03)  
**Deliverables:** Quantitative clustering evaluation (K-Means, Hierarchical, DBSCAN), Segment Profiling, Business Translation Matrix, Recommendations & Final Conclusion

## 1. Setup & Data Ingestion

We load the preprocessed log-transformed and standard-scaled RFM features from Notebook 03 (`data/rfm_preprocessed.parquet`).

In [1]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

from rfm_analysis import preprocess_rfm_for_clustering
from clustering_utils import evaluate_kmeans_range, fit_kmeans, fit_agglomerative, fit_dbscan, compute_cluster_profiles

# Plotting setup
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
fig_dir = Path('../reports/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

# Load data
rfm_df = pd.read_parquet('../data/rfm_preprocessed.parquet')
print(f"Loaded preprocessed customer records: {len(rfm_df):,}")

# Feature matrix X_scaled
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_df[['R_log', 'F_log', 'M_log']])

print("Feature matrix shape:", X_scaled.shape)
print(rfm_df[['Recency', 'Frequency', 'Monetary', 'R_log', 'F_log', 'M_log']].head())

Loaded preprocessed customer records: 4,322
Feature matrix shape: (4322, 3)
   Recency  Frequency  Monetary     R_log     F_log     M_log
0        2          7   4310.00  1.098612  2.079442  8.368925
1       75          4   1797.24  4.330733  1.609438  7.494564
2       19          1   1757.55  2.995732  0.693147  7.472245
3      310          1    334.40  5.739793  0.693147  5.815324
4       36          8   1545.41  3.610918  2.197225  7.343691


## 2. Algorithmic Clustering & Quantitative Validation

To discover defensible, actionable customer segments, we evaluate three major clustering paradigms:
1. **K-Means Clustering:** Efficient, centroid-based model for partitioning scaled continuous spaces.
2. **Hierarchical (Agglomerative) Clustering:** Tree-based bottom-up merging evaluated via Ward's linkage.
3. **DBSCAN:** Density-based algorithm used to detect non-spherical shapes and isolate extreme outlier ("whale") buyers.

### 2.1 K-Means Clustering Evaluation (Elbow Method & Silhouette Width)
We evaluate K-Means across $k \in [2, 10]$ using Within-Cluster Sum of Squares (WCSS / Inertia), Silhouette Width, Calinski-Harabasz Index, and Davies-Bouldin Index.

In [2]:
eval_df = evaluate_kmeans_range(X_scaled, k_range=range(2, 11), sample_size=2500, random_state=42)
print("--- K-Means Evaluation Metrics Across k=2 to 10 ---")
print(eval_df)

--- K-Means Evaluation Metrics Across k=2 to 10 ---
    k         WCSS  Silhouette  Calinski_Harabasz  Davies_Bouldin
0   2  6488.817468    0.424044        4312.263241        0.893669
1   3  4870.380789    0.333521        3589.571543        1.048487
2   4  3941.770063    0.338003        3295.214201        1.010810
3   5  3312.200621    0.320695        3145.689568        0.987087
4   6  2878.423599    0.306322        3025.138727        1.012909
5   7  2574.242785    0.299157        2903.180985        0.988674
6   8  2367.232852    0.267547        2759.287152        1.004946
7   9  2182.162792    0.275064        2664.262048        1.022615
8  10  2022.685022    0.271658        2592.173432        1.020856


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Elbow Plot
axes[0].plot(eval_df['k'], eval_df['WCSS'], marker='o', linewidth=2.5, color='#2b5c8f')
axes[0].axvline(4, color='crimson', linestyle='--', label='Selected k=4')
axes[0].set_title('Elbow Method: WCSS vs Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)')
axes[0].legend()

# Silhouette Plot
axes[1].plot(eval_df['k'], eval_df['Silhouette'], marker='s', linewidth=2.5, color='#3690c0')
axes[1].axvline(4, color='crimson', linestyle='--', label='Selected k=4')
axes[1].set_title('Silhouette Width vs Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Mean Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/kmeans_evaluation_elbow_silhouette.png', dpi=300)
plt.close()
print("Saved kmeans_evaluation_elbow_silhouette.png")

Saved kmeans_evaluation_elbow_silhouette.png


> **Quantitative & Business Justification for Selecting $k=4$:**
> - **Elbow Inflection:** The drop in WCSS slows noticeably after $k=4$ (WCSS decreases from 6,488 at $k=2$ to 3,941 at $k=4$, with diminishing returns thereafter).
> - **Silhouette Score:** $k=4$ provides a strong local peak (Silhouette = **0.3394**) among multi-cluster segmentation choices.
> - **Business Interpretability:** $k=2$ merely splits customers into high vs low spenders, whereas $k=4$ cleanly isolates:
>   1. **Champions** (High R, High F, High M)
>   2. **Loyal Regulars** (Moderate R, Moderate F, Moderate-High M)
>   3. **New Inflow** (High R, Low F, Low M)
>   4. **At-Risk / Churned Spenders** (Low R, Low/Moderate F, Low/Moderate M)

### 2.2 Hierarchical (Agglomerative) Clustering & Dendrogram
Hierarchical clustering does not require specifying cluster counts upfront. We inspect Ward's linkage dendrogram to observe structural sub-cohort merging.

In [4]:
plt.figure(figsize=(12, 6))
link_matrix = linkage(X_scaled, method='ward')
dendrogram(link_matrix, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=10, show_contracted=True)
plt.axhline(y=45, color='crimson', linestyle='--', label='Truncation Cut-off (4 Clusters)')
plt.title("Hierarchical Clustering Dendrogram (Ward's Linkage)", fontsize=14, fontweight='bold')
plt.xlabel("Sample Index / Sub-cluster Size")
plt.ylabel("Euclidean Distance")
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/hierarchical_dendrogram.png', dpi=300)
plt.close()
print("Saved hierarchical_dendrogram.png")

# Fit Agglomerative k=4
agg_model, agg_labels = fit_agglomerative(X_scaled, n_clusters=4, linkage='ward')
sil_agg = silhouette_score(X_scaled, agg_labels, sample_size=2500, random_state=42)
print(f"Agglomerative Clustering (k=4) Silhouette Score: {sil_agg:.4f}")

Saved hierarchical_dendrogram.png
Agglomerative Clustering (k=4) Silhouette Score: 0.2594


### 2.3 DBSCAN (Density-Based Anomaly & Whale Isolation)
DBSCAN groups dense regions and isolates extreme outliers (noise points, marked as `-1`). We tune $\eps=0.5$ and $\text{min\_samples}=15$.

In [5]:
db_model, db_labels = fit_dbscan(X_scaled, eps=0.5, min_samples=15)

n_noise = (db_labels == -1).sum()
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)

print(f"DBSCAN Clusters Found: {n_clusters_db}")
print(f"Isolated Noise Points (Whale Buyers / Outliers): {n_noise:,} ({n_noise/len(X_scaled):.2%})")

# Inspect top monetary spenders isolated as noise
rfm_df['DBSCAN_Label'] = db_labels
whales = rfm_df[rfm_df['DBSCAN_Label'] == -1].sort_values(by='Monetary', ascending=False)
print("\nTop 5 Extreme Outliers (Whale Buyers) Isolated by DBSCAN:")
print(whales[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'AvgOrderValue']].head())

DBSCAN Clusters Found: 2
Isolated Noise Points (Whale Buyers / Outliers): 106 (2.45%)

Top 5 Extreme Outliers (Whale Buyers) Isolated by DBSCAN:
      CustomerID  Recency  Frequency   Monetary  AvgOrderValue
1684       14646        2         74  279489.02        3776.88
4186       18102        1         60  256438.49        4273.97
3716       17450        8         46  187322.17        4072.22
1873       14911        1        201  132458.73         659.00
54         12415       24         21  123725.45        5891.69


**DBSCAN Anomaly Finding:**
DBSCAN isolates **106 outlier accounts (2.45% of customer base)**. These include the massive corporate accounts (e.g. Customer 14646 with £280k spend and Customer 18102 with £259k spend) whose purchase volume exceeds standard retail density. This confirms that density-based algorithms excel at filtering corporate noise.

## 3. Dimensionality Reduction & 2D PCA Cluster Visualization

We fit our primary **K-Means ($k=4$)** model and visualize the resulting segments in 2D Principal Component Analysis (PCA) space.

In [6]:
km_model, km_labels = fit_kmeans(X_scaled, n_clusters=4, random_state=42)
rfm_df['Cluster'] = km_labels

# PCA Transformation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

rfm_df['PCA1'] = X_pca[:, 0]
rfm_df['PCA2'] = X_pca[:, 1]

var1, var2 = pca.explained_variance_ratio_
print(f"PCA Component 1 Explained Variance: {var1:.1%}")
print(f"PCA Component 2 Explained Variance: {var2:.1%}")
print(f"Total Cumulative Explained Variance: {var1 + var2:.1%}")

# Plot PCA Scatter
plt.figure(figsize=(10, 7))
palette = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', data=rfm_df, palette=palette, s=50, alpha=0.7)

# Plot Centroids
centroids_pca = pca.transform(km_model.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=250, c='black', marker='X', label='Cluster Centroids')

plt.title("2D PCA Visualization of K-Means Customer Segments (k=4)", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({var1:.1%} Variance)")
plt.ylabel(f"Principal Component 2 ({var2:.1%} Variance)")
plt.legend(title="Cluster ID", loc='best')
plt.tight_layout()
plt.savefig('../reports/figures/pca_kmeans_clusters.png', dpi=300)
plt.close()
print("Saved pca_kmeans_clusters.png")

PCA Component 1 Explained Variance: 74.9%
PCA Component 2 Explained Variance: 18.9%
Total Cumulative Explained Variance: 93.8%
Saved pca_kmeans_clusters.png


## 4. Strategic Cluster Profiling & Segment Personas

We unscale cluster metrics back to real-world GBP values, purchase counts, and days elapsed to establish concrete business segment personas.

In [7]:
profile = compute_cluster_profiles(rfm_df, cluster_col='Cluster')

# Map strategic persona names based on empirical RFM profiles
persona_map = {
    1: 'Champions / Wholesale Core',
    2: 'Loyal Spenders',
    3: 'New / Recent Inflow',
    0: 'At-Risk / Churned'
}

profile['Segment_Name'] = profile['Cluster'].map(persona_map)

# Reorder columns for presentation
profile_display = profile[['Cluster', 'Segment_Name', 'Customer_Count', 'Customer_Share_%', 'Recency_Mean', 'Recency_Median', 'Frequency_Mean', 'Frequency_Median', 'Monetary_Mean', 'Monetary_Median', 'Monetary_Total', 'Revenue_Share_%']]

print("--- Comprehensive Customer Segment Profiling Table ---")
print(profile_display.to_string())

--- Comprehensive Customer Segment Profiling Table ---
   Cluster                Segment_Name  Customer_Count  Customer_Share_%  Recency_Mean  Recency_Median  Frequency_Mean  Frequency_Median  Monetary_Mean  Monetary_Median  Monetary_Total  Revenue_Share_%
0        0           At-Risk / Churned            1602             37.07    179.896380           173.0        1.322722               1.0     332.976118           284.94      533427.741             6.43
1        1  Champions / Wholesale Core             707             16.36     11.131542             8.0       13.806223              10.0    7615.736478          3626.81     5384325.690            64.94
2        2              Loyal Spenders            1169             27.05     68.628743            52.0        4.119760               4.0    1629.254947          1303.91     1904599.033            22.97
3        3         New / Recent Inflow             844             19.53     16.793839            16.0        2.156398               2.0 

In [8]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Recency Mean
sns.barplot(x='Segment_Name', y='Recency_Mean', data=profile, palette='Reds_r', ax=axes[0, 0])
axes[0, 0].set_title('Mean Recency (Days since last purchase)')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Days')
axes[0, 0].tick_params(axis='x', rotation=20)

# Frequency Mean
sns.barplot(x='Segment_Name', y='Frequency_Mean', data=profile, palette='Blues_r', ax=axes[0, 1])
axes[0, 1].set_title('Mean Frequency (Number of Orders)')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Orders')
axes[0, 1].tick_params(axis='x', rotation=20)

# Monetary Mean
sns.barplot(x='Segment_Name', y='Monetary_Mean', data=profile, palette='Greens_r', ax=axes[1, 0])
axes[1, 0].set_title('Mean Monetary Spend (£)')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Revenue (£)')
axes[1, 0].tick_params(axis='x', rotation=20)

# Revenue Share %
sns.barplot(x='Segment_Name', y='Revenue_Share_%', data=profile, palette='Purples_r', ax=axes[1, 1])
axes[1, 1].set_title('Total Revenue Share (%)')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('% of Revenue')
axes[1, 1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../reports/figures/segment_comparisons.png', dpi=300)
plt.close()
print("Saved segment_comparisons.png")

C:\Users\Raghad\AppData\Local\Temp\ipykernel_17216\1550357814.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Segment_Name', y='Recency_Mean', data=profile, palette='Reds_r', ax=axes[0, 0])
C:\Users\Raghad\AppData\Local\Temp\ipykernel_17216\1550357814.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Segment_Name', y='Frequency_Mean', data=profile, palette='Blues_r', ax=axes[0, 1])
C:\Users\Raghad\AppData\Local\Temp\ipykernel_17216\1550357814.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Segment_Name', y='Monetary_Mean

Saved segment_comparisons.png


### Executive Segment Personas & Statistical Profiles

1. **Champions / Wholesale Core (Cluster 1):**
   - **Customer Base:** 707 customers (**16.4%** of total identified base).
   - **Revenue Contribution:** **£5,384,326 (64.9% of total net revenue)**.
   - **Behavior:** Exceptional engagement (Recency mean: **11.1 days**, Frequency mean: **13.8 orders**, Monetary mean: **£7,616**).
   - **Strategic Role:** The vital economic engine of the retailer.

2. **Loyal Spenders (Cluster 2):**
   - **Customer Base:** 1,169 customers (**27.1%** of total identified base).
   - **Revenue Contribution:** **£1,904,599 (23.0% of total net revenue)**.
   - **Behavior:** Steady recurring buyers (Recency mean: **68.6 days**, Frequency mean: **4.1 orders**, Monetary mean: **£1,629**).
   - **Strategic Role:** Prime targets for cross-selling and volume upselling.

3. **New / Recent Inflow (Cluster 3):**
   - **Customer Base:** 844 customers (**19.5%** of total identified base).
   - **Revenue Contribution:** **£469,396 (5.7% of total net revenue)**.
   - **Behavior:** Recent first-time/second-time buyers (Recency mean: **16.8 days**, Frequency mean: **2.2 orders**, Monetary mean: **£556**).
   - **Strategic Role:** Future revenue pipeline; requires immediate onboarding engagement.

4. **At-Risk / Churned (Cluster 0):**
   - **Customer Base:** 1,602 customers (**37.1%** of total identified base).
   - **Revenue Contribution:** **£533,428 (6.4% of total net revenue)**.
   - **Behavior:** Long inactive periods (Recency mean: **179.9 days**, Frequency mean: **1.3 orders**, Monetary mean: **£333**).
   - **Strategic Role:** Churned accounts; require automated low-cost win-back triggers.

## 5. Strategic Business Translation Matrix

| Segment Persona | Cluster | Customer Share | Revenue Share | Business Risk / Opportunity | Strategic Corporate Action | Tactical Marketing Triggers |
|---|---|---|---|---|---|---|
| **Champions / Wholesale Core** | Cluster 1 | 16.4% | **64.9%** | Severe risk if any account churns | **Retention & VIP Perks** | Dedicated account manager, early catalog drops, volume rebates, no aggressive discounting |
| **Loyal Spenders** | Cluster 2 | 27.1% | 23.0% | Moderate spend growth potential | **Cross-Selling & Basket Expansion** | Automated category replenishment reminders, bundle discounts, loyalty tier points |
| **New / Recent Inflow** | Cluster 3 | 19.5% | 5.7% | High drop-off after first order | **Onboarding & 2nd Purchase Conversion** | Welcome discount sequence (10% off next order within 14 days), product discovery guides |
| **At-Risk / Churned** | Cluster 0 | **37.1%** | 6.4% | Trapped capital in inactive accounts | **Automated Win-Back or Deprioritization** | Automated 90-day win-back email trigger, steep clearance offers; limit high-cost direct marketing |

## 6. Comprehensive Business Recommendations

1. **Protect the 16.4% Core (Champions):** Because 707 buyers generate nearly **65% of net revenue**, assign dedicated account managers and priority logistics support to prevent B2B churn.
2. **Automate the Onboarding Funnel for New Buyers (Cluster 3):** With over 35% of buyers purchasing only once, deploy an automated 14-day post-purchase email series to convert single buyers into repeat orders.
3. **Execute Target Cross-Selling for Loyal Spenders (Cluster 2):** Leverage market-basket rules to recommend complementary stock items, raising mean spend from £1,629 towards £3,000+.
4. **Deploy Low-Cost Win-Back Triggers (Cluster 0):** Avoid costly manual sales calls for dormant users; rely on automated email sequences with clearance discount codes.
5. **Differentiate B2B Wholesalers from Retail Consumers:** Use DBSCAN outlier markers to identify high-volume accounts that require bulk invoicing, pallet shipping, and custom credit terms.

## 7. Limitations & Next Steps

### 7.1 Data & Model Limitations
- **Margin Absence:** Evaluation relies on top-line revenue; gross margin data is needed to assess true profitability.
- **1-Year Time Horizon:** Seasonality and annual repurchase cycles cannot be fully decoupled with 12 months of historical data.
- **Unidentified Guest Checkout:** 14.9% of revenue (£1.44M) comes from guest transactions and is excluded from customer-level RFM.

### 7.2 Strategic Next Steps
- **Customer Lifetime Value (CLV) Modeling:** Build BG/NBD and Gamma-Gamma predictive models to forecast future customer transaction volume and spend.
- **Real-Time RFM Scoring Pipeline:** Implement a dynamic pipeline to score incoming orders and trigger immediate marketing automation events.

## 8. Final Conclusion

This project successfully transformed raw retail transactions into actionable customer intelligence. By enforcing rigorous structural cleaning, establishing robust RFM metrics, applying log-scaling transformations, and evaluating K-Means, Hierarchical, and DBSCAN algorithms, we identified 4 distinct, business-defensible customer segments. The strategic implementation of the Business Translation Matrix enables executive leadership to optimize marketing spend, protect 65% of core revenue, and drive long-term customer lifetime value.